# Basic SDPA operation using cudnn FE
This notebook shows how a sdpa operation with causal mask can be done using cudnn.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/https://github.com/NVIDIA/cudnn-frontend/tree/main/samples/python/convolutions/00_basic_convolutions.ipynb)

## Prerequisites for running on Colab
This notebook requires an NVIDIA GPU A100 or newer. If `nvidia-smi` fails, go to Runtime -> Change runtime type -> Hardware accelerator and confirm a GPU is selected.

In [ ]:
!#nvidia-smi

If running on Colab, you will need to install the cudnn python interface.

In [ ]:
!# export CUDA_VERSION="12.3"
!# pip install nvidia-cudnn-cu12
!# conda install -y -c nvidia cuda-nvcc="${CUDA_VERSION}" cuda-libraries-dev="${CUDA_VERSION}"
!# CUDNN_PATH=`pip show nvidia-cudnn-cu12  | grep Location | cut -d":" -f2 | args`/nvidia/cudnn pip install git+https://github.com/NVIDIA/cudnn-frontend.git
!# pip3 install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu121

#### General Setup
We are going to call the cudnn through torch in this example. In general any dlpack tensor should work.
cudnn handle is a per device handle used to initialize cudnn context.


In [ ]:
import cudnn
import torch
import math

handle = cudnn.create_handle()

In [ ]:
#### Create input tensors and calculate reference

b = 2 # batch size

s_q  = 1024 # query sequence length
s_kv = 1024 # key+value sequence length

d_qk = 64   # query+key embedding dimension per head
d_v  = 64   # value embedding dimension per head

h_q = 6 # Query heads
h_k = 6 # Key heads. This changes for Group, Multi query
h_v = 6 # Value heads

shape_q = (b, h_q, s_q, d_qk)
shape_k = (b, h_k, s_kv, d_qk)
shape_v = (b, h_v, s_kv, d_v)
shape_o = (b, h_q, s_q, d_v)

stride_q = (s_q  * h_q * d_qk, d_qk, h_q * d_qk, 1)
stride_k = (s_kv * h_k * d_qk, d_qk, h_k * d_qk, 1)
stride_v = (s_kv * h_v * d_v,  d_v,  h_v * d_v , 1)
stride_o = (s_q  * h_q * d_v,  d_v,  h_q * d_v , 1)


attn_scale = 0.125

In [ ]:
if  cudnn.backend_version() < 8903:
   print("Not Supported in cudnn v8.9.3 or below")

In [ ]:
qkv_num_elems = math.prod(shape_q) + math.prod(shape_k) + math.prod(shape_v)

q_offset = 0
k_offset = b * s_q * h_q * d_qk
v_offset = k_offset + b * s_kv * h_k * d_qk

qkv_gpu = torch.randn(qkv_num_elems, dtype=torch.bfloat16, device="cuda") - 0.5

q_gpu = torch.as_strided(qkv_gpu, shape_q, stride_q, storage_offset=q_offset)
k_gpu = torch.as_strided(qkv_gpu, shape_k, stride_k, storage_offset=k_offset)
v_gpu = torch.as_strided(qkv_gpu, shape_v, stride_v, storage_offset=v_offset)

o_gpu     = torch.empty(b * h_q * s_q * d_v, dtype=torch.bfloat16, device="cuda").as_strided(shape_o, stride_o)
stats_gpu = torch.empty(b, h_q, s_q, 1, dtype=torch.float32, device="cuda")


In [ ]:
graph = cudnn.pygraph(
    io_data_type=cudnn.data_type.BFLOAT16,
    intermediate_data_type=cudnn.data_type.FLOAT,
    compute_data_type=cudnn.data_type.FLOAT,
)

q = graph.tensor_like(q_gpu)
k = graph.tensor_like(k_gpu)
v = graph.tensor_like(v_gpu)

o, stats = graph.sdpa(
    name="sdpa",
    q=q, k=k, v=v,
    is_inference=False,
    attn_scale=attn_scale,
    use_causal_mask=True,
)

o.set_output(True).set_dim(shape_o).set_stride(stride_o)
stats.set_output(True).set_data_type(cudnn.data_type.FLOAT)


Q_UID    = 0
K_UID    = 1
V_UID    = 2
O_UID    = 3
STATS_UID = 4

q.set_uid(Q_UID)
k.set_uid(K_UID)
v.set_uid(V_UID)
o.set_uid(O_UID)
stats.set_uid(STATS_UID)

In [ ]:
graph.validate()
graph.build_operation_graph()
graph.create_execution_plans([cudnn.heur_mode.A, cudnn.heur_mode.FALLBACK])
graph.check_support()
graph.build_plans()

In [ ]:
variant_pack = {
    Q_UID: q_gpu,
    K_UID: k_gpu,
    V_UID: v_gpu,
    O_UID: o_gpu,
    STATS_UID: stats_gpu,
}

In [ ]:
workspace = torch.empty(graph.get_workspace_size(), device="cuda", dtype=torch.uint8)
graph.execute(variant_pack, workspace)
torch.cuda.synchronize()